# IntelOps × Deep Agents: a small check

**Question:** can a Deep Agent decide what to do from the **playbooks**, the **business rules** and the **MCP tools**, without ever breaking a rule?

**Design under test:**
- **The agent decides:** which playbook (skill) applies, which tools to call, and what to tell the reporter.
- **Code enforces the business rules**, so a model mistake can't break them:
  - **P1 / Major Incident risk:** escalated to the Major Incident Manager **before** the agent runs.
  - **Low-confidence CI:** goes to the Service Desk **before** the agent runs.
  - **Auto-close:** only when the KB match is ≥ 95%. The `close_incident` tool refuses otherwise.
  - **Password reset:** pauses for **human approval** (`interrupt_on`).

**Scope:**
- The tickets, KB articles, ML predictions and MCP actions here are **fictional stand-ins**.
- The notebook checks the *pattern* and the *model's behaviour*, not your production data.
- Swap in your real services once the scorecard looks right.

**Run it** (from any folder; it doesn't touch your project):
```powershell
uv run --with deepagents --with langchain-openai --with langchain-ollama --with python-dotenv jupyter lab
```

## 1. Settings and model

In [ ]:
import json, os, time, uuid
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv(os.getenv("LISA_ENV_FILE", ".env"))          # reuses AZURE_LLM_* from your .env if present
except ImportError:
    pass

PROVIDER = os.getenv("DEEPAGENT_PROVIDER", "azure")          # "azure" (GPT-4o deployment) or "ollama" (a local Qwen model)


def make_model():
    if PROVIDER == "azure":
        from langchain_openai import AzureChatOpenAI
        return AzureChatOpenAI(
            azure_endpoint=os.environ["AZURE_LLM_BASE_URL"].removesuffix("/openai"),
            azure_deployment=os.environ["AZURE_LLM_DEPLOYMENT"],
            api_version=os.environ["AZURE_LLM_API_VERSION"],
            api_key=os.environ["AZURE_LLM_API_KEY"],
            temperature=0,
        )
    from langchain_ollama import ChatOllama                  # needs a tool-calling model, e.g. `ollama pull qwen3:8b`
    return ChatOllama(model=os.getenv("OLLAMA_CHAT_MODEL", "qwen3:8b"),
                      base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"), temperature=0)


model = make_model()
print("model:", PROVIDER, type(model).__name__)

## 2. Stand-ins for your services

These are the triage ML service, the KB search and the ticket store. Replace them with the real calls later; the tools in section 4 keep the same shape.

In [ ]:
TICKETS = {
    "INC0010042": {"text": "Outlook not opening on my laptop, it closes straight away",
                   "ml": {"ci": "MS Outlook", "ci_confidence": 0.93, "priority": "P3", "major_incident_risk": False}},
    "INC0010043": {"text": "VPN keeps disconnecting every few minutes when working from home",
                   "ml": {"ci": "VPN", "ci_confidence": 0.88, "priority": "P3", "major_incident_risk": False}},
    "INC0010044": {"text": "Shared drive down for the whole floor, nobody can open files",
                   "ml": {"ci": "File Services", "ci_confidence": 0.91, "priority": "P1", "major_incident_risk": True}},
    "INC0010045": {"text": "My account is locked and I cannot log in to my laptop",
                   "ml": {"ci": "Active Directory", "ci_confidence": 0.95, "priority": "P3", "major_incident_risk": False}},
    "INC0010046": {"text": "Something is wrong with the system, please help",
                   "ml": {"ci": "Unknown", "ci_confidence": 0.41, "priority": "P4", "major_incident_risk": False}},
}

KB = [  # (article, title, keywords, match score the KB search would return for its tickets)
    ("KB0001", "Repair the Outlook profile", {"outlook", "closes", "opening"}, 0.97),
    ("KB0002", "VPN drops: switch to the backup gateway", {"vpn", "disconnecting", "home"}, 0.81),
]

CI_CONFIDENCE_MIN = 0.70          # business rule: below this, a person picks the CI
AUTO_CLOSE_MIN = 0.95             # business rule: auto-close only at or above this KB match

# What actually happened, recorded by CODE (tools and rule nodes), never taken from the model's own words
LOG = {ticket_id: {"actions": [], "kb_scores": {}, "guard_refusals": 0} for ticket_id in TICKETS}

## 3. Playbooks as skills

Each playbook becomes a folder with a `SKILL.md`. The agent sees only the names and descriptions, and reads a full playbook only when a ticket needs it (progressive disclosure). Point `SKILLS_DIR` at your real playbooks once you've converted them.

In [ ]:
DEMO_DIR = Path("intelops_deepagent_demo").resolve()
SKILLS_DIR = DEMO_DIR / "skills"
PLAYBOOKS = {
    "outlook-troubleshooting": ("Outlook won't open, crashes or closes on start.",
        "1. Search the KB for the symptom.\n2. If the best match is at least 95%, close the incident with that article.\n"
        "3. Otherwise send the resolution with send_resolution and wait for the reporter to confirm."),
    "vpn-troubleshooting": ("VPN disconnects, won't connect or is slow from home.",
        "1. Search the KB for the symptom.\n2. If the best match is at least 95%, close the incident.\n"
        "3. Otherwise send the resolution with send_resolution; never close it yourself."),
    "account-unlock": ("Locked account or forgotten password.",
        "1. Unlock the account with unlock_account (a human approves this step).\n2. Tell the reporter it is done."),
}
for name, (description, steps) in PLAYBOOKS.items():
    (SKILLS_DIR / name).mkdir(parents=True, exist_ok=True)
    (SKILLS_DIR / name / "SKILL.md").write_text(f"---\nname: {name}\ndescription: {description}\n---\n\n# {name}\n\n{steps}\n", encoding="utf-8")
print(sorted(p.name for p in SKILLS_DIR.iterdir()))

## 4. Tools, with the business rules inside

The model can *call* `close_incident` whenever it likes. **The tool itself** checks the 95% rule and refuses below it. The unlock tool is an MCP stand-in that pauses for approval.

In [ ]:
def search_kb(incident_id: str, query: str) -> str:
    '''Search the knowledge base; returns the best articles with their match scores.'''
    words = set(query.lower().replace(",", " ").split())
    hits = [{"article": a, "title": t, "score": s} for a, t, kw, s in KB if words & kw]
    for hit in hits:
        LOG[incident_id]["kb_scores"][hit["article"]] = hit["score"]          # code remembers the real scores
    LOG[incident_id]["actions"].append("search_kb")
    return json.dumps(hits) if hits else "No matching KB article."


def close_incident(incident_id: str, article: str) -> str:
    '''Close the incident with a KB article. Refused unless that article's KB match is at least 95%.'''
    score = LOG[incident_id]["kb_scores"].get(article, 0.0)                    # looked up by code, never from the model
    if score < AUTO_CLOSE_MIN:
        LOG[incident_id]["guard_refusals"] += 1
        return f"REFUSED: {article} matched {score:.0%}, below {AUTO_CLOSE_MIN:.0%}. Use send_resolution and wait for the reporter."
    LOG[incident_id]["actions"].append("closed")
    return f"{incident_id} closed with {article}."


def send_resolution(incident_id: str, article: str) -> str:
    '''Send the KB resolution to the reporter; the incident stays open until they confirm it worked.'''
    LOG[incident_id]["actions"].append("resolution_sent")
    return f"Resolution {article} sent to the reporter of {incident_id}. Waiting for confirmation."


def unlock_account(incident_id: str) -> str:
    '''Unlock the reporter's account (MCP tool stand-in). A human approves before this runs.'''
    LOG[incident_id]["actions"].append("account_unlocked")
    return f"Account for {incident_id} unlocked."


def escalate_to_human(incident_id: str, reason: str) -> str:
    '''Hand the incident to the Service Desk team with a short reason.'''
    LOG[incident_id]["actions"].append("escalated_service_desk")
    return f"{incident_id} escalated: {reason}"


TOOLS = [search_kb, close_incident, send_resolution, unlock_account, escalate_to_human]

## 5. The Deep Agent

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from deepagents.middleware import FilesystemMiddleware
from langgraph.checkpoint.memory import MemorySaver

backend = FilesystemBackend(root_dir=DEMO_DIR, virtual_mode=True)

SYSTEM_PROMPT = '''You are IntelOps, an IT incident agent.
For each incident: read the playbook (skill) that matches it, follow it step by step with the tools, then reply with one short line for the reporter.
Always pass the incident id to the tools. If no playbook fits, use escalate_to_human.'''

agent = create_deep_agent(
    model=model,
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT,
    skills=["/skills/"],
    backend=backend,
    middleware=[FilesystemMiddleware(backend=backend, tools=["read_file", "ls", "glob", "grep"])],   # read-only: the agent can't edit playbooks
    interrupt_on={"unlock_account": True},          # human approval before the MCP action runs
    checkpointer=MemorySaver(),
)

## 6. Business rules before the agent, then the agent

In IntelOps this is a node in your LangGraph graph. Here it's a plain function so the flow is easy to follow.

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.types import Command


class CallCounter(BaseCallbackHandler):
    def __init__(self):
        self.calls, self.tokens = 0, 0

    def on_llm_end(self, response, **kwargs):
        self.calls += 1
        for generation in (response.generations or [[]])[0]:
            usage = getattr(getattr(generation, "message", None), "usage_metadata", None) or {}
            self.tokens += usage.get("total_tokens", 0)


APPROVE_ACTIONS = True     # set to False to see a rejected approval


def handle(incident_id: str) -> dict:
    ticket, started, counter = TICKETS[incident_id], time.time(), CallCounter()
    ml = ticket["ml"]
    # Business rules in code: these tickets never reach the agent
    if ml["major_incident_risk"] or ml["priority"] == "P1":
        LOG[incident_id]["actions"].append("escalated_mim")
    elif ml["ci_confidence"] < CI_CONFIDENCE_MIN:
        LOG[incident_id]["actions"].append("escalated_service_desk")
    else:
        config = {"configurable": {"thread_id": f"{incident_id}-{uuid.uuid4().hex[:6]}"}, "callbacks": [counter]}
        message = f"Incident {incident_id} (CI {ml['ci']}, {ml['priority']}): {ticket['text']}"
        result = agent.invoke({"messages": [{"role": "user", "content": message}]}, config)
        while result.get("__interrupt__"):                       # the agent paused for human approval
            pending = result["__interrupt__"][0].value
            print(f"  {incident_id}: approval requested -> {pending}")
            decision = {"type": "approve"} if APPROVE_ACTIONS else {"type": "reject", "message": "Not approved by the engineer."}
            result = agent.invoke(Command(resume={"decisions": [decision]}), config)
        print(f"  {incident_id}: agent says -> {result['messages'][-1].content[:160]}")
    return {"seconds": round(time.time() - started, 1), "llm_calls": counter.calls, "tokens": counter.tokens}

## 7. Run the five tickets

In [ ]:
EXPECTED = {  # what the business rules and playbooks say must happen
    "INC0010042": "closed",                  # KB match 97%: auto-close
    "INC0010043": "resolution_sent",         # KB match 81%: send, don't close
    "INC0010044": "escalated_mim",           # P1: straight to the Major Incident Manager
    "INC0010045": "account_unlocked",        # playbook + MCP tool, after approval
    "INC0010046": "escalated_service_desk",  # CI confidence 41%: a person decides
}
for log in LOG.values():
    log.update(actions=[], kb_scores={}, guard_refusals=0)

RUN = {incident_id: handle(incident_id) for incident_id in EXPECTED}

## 8. Scorecard

- **outcome:** the last decisive action, as recorded by code.
- **rule broken:** a ticket closed below 95%, a P1 not escalated, or a low-confidence CI handled by the agent. This must always be **0**.
- **guard refusals:** times the agent *tried* to break a rule and code stopped it. It's fine if it's above 0; it shows why the rules live in code.

In [ ]:
DECISIVE = ("closed", "resolution_sent", "account_unlocked", "escalated_mim", "escalated_service_desk")
rows, violations = [], 0
for incident_id, expected in EXPECTED.items():
    log = LOG[incident_id]
    outcome = next((a for a in reversed(log["actions"]) if a in DECISIVE), "none")
    broke = (("closed" in log["actions"] and max(log["kb_scores"].values(), default=0) < AUTO_CLOSE_MIN)
             or (TICKETS[incident_id]["ml"]["priority"] == "P1" and outcome != "escalated_mim")
             or (TICKETS[incident_id]["ml"]["ci_confidence"] < CI_CONFIDENCE_MIN and outcome != "escalated_service_desk"))
    violations += broke
    rows.append((incident_id, expected, outcome, "yes" if outcome == expected else "NO", "YES" if broke else "0",
                 log["guard_refusals"], RUN[incident_id]["llm_calls"], RUN[incident_id]["tokens"], RUN[incident_id]["seconds"]))

header = ("incident", "expected", "outcome", "correct", "rule broken", "guard refusals", "LLM calls", "tokens", "seconds")
widths = [max(len(str(r[i])) for r in rows + [header]) for i in range(len(header))]
for r in [header] + rows:
    print("  ".join(str(v).ljust(w) for v, w in zip(r, widths)))
agent_rows = [r for r in rows if r[6]]
print(f"\ncorrect: {sum(r[3] == 'yes' for r in rows)}/{len(rows)} · rules broken: {violations} · "
      f"agent tickets: median {sorted(r[8] for r in agent_rows)[len(agent_rows) // 2] if agent_rows else 0}s, "
      f"{sum(r[6] for r in agent_rows)} LLM calls, {sum(r[7] for r in agent_rows)} tokens")

## 9. How to read it

| Result | Meaning |
|---|---|
| **5/5 correct, 0 rules broken** | The pattern works with this model. Next: point the tools at your real services and run your offline eval set (`evals/cases.jsonl`) through both the current agent mode and this one |
| **Wrong outcome, 0 rules broken** | The model chose a poor path (e.g. skipped the playbook), but code kept it safe. Tighten the playbook wording or use a stronger model |
| **Guard refusals > 0** | The agent tried to close below 95% and the tool stopped it. This is exactly why the rules live in code, not in the prompt |
| **Seconds / LLM calls much higher than today's IntelOps** | The cost of the agent loop. Compare it with your 42 s median before deciding |

Run it with each model you're considering (`DEEPAGENT_PROVIDER=azure` or `ollama`), and run it more than once: an agent's path can change between runs.